# Spectral animations .py file

Mental note:
This was generated with AI assistance, so labels are likely hallucinated, focus in behaviour of the data



In [ ]:
# ============================================================
# @title Polar unwrap animation — det(I - z Σ_N^{exp(i θ_op)})
#
# This is the actual object from spectral.py, now shown in a
# polar (arg z, |z|) unrolled view.
#
# Visual:
#   x-axis = arg(z) = φ in z = r exp(i φ)     ← angle of z in the complex plane
#   y-axis = |z| = r                            ← radial distance
#   color  = arg det(I - z Σ_N^{exp(i θ_op)}) ← phase of the Fredholm det
#   dots   = zeros z_j(θ_op) = λ_j^{-exp(i θ_op)} plotted in (arg z_j, |z_j|)
#
# What was wrong in the ChatGPT cell
# -----------------------------------
# That cell showed arg(z) as the *animation variable*, sweeping θ = arg(z)
# while r ran on the y-axis.  That is just scanning the plain det(I - z Σ)
# over the z-plane in polar coordinates — no spectral rotation.
#
# The spectral rotation θ_op rotates the *operator exponent*: Σ^{exp(i θ_op)}.
# Its zeros z_j(θ_op) = λ_j^{-exp(i θ_op)} orbit the complex plane as θ_op
# varies, and that orbit is what this animation shows.
#
# Key orbit landmarks (all exact, zero error):
#   θ_op = 0°   →  z_j = 1/λ_j       (real, positive, on φ=0 line)
#   θ_op = 90°  →  |z_j| = 1 for all j (every zero lands on the unit circle)
#   θ_op = 180° →  z_j = λ_j         (real, positive, inside unit circle)
#   θ_op = 270° →  |z_j| = 1 again
#
# Animation modes
# ---------------
# ANIMATE_OVER = 'theta_op'  — sweep θ_op from 0 → 2π, fixed N
#                              The zeros orbit; phase field breathes.
# ANIMATE_OVER = 'N'         — grow N, fixed θ_op = 0
#                              Same intent as the original cell, but computed
#                              correctly via Newton's identities (no branch cuts).
#
# Computation (identical to spectral.py)
# ----------------------------------------
# Uses elementary_cx + poly_logdet (Newton's identities + Horner) — the same
# routines spectral.py uses for its 3D surface, so results are identical.
# No complex-log-sum branch-cut artifacts.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from math import gcd
from IPython.display import HTML

# ── Config ────────────────────────────────────────────────────────────────────

ANIMATE_OVER = 'theta_op'   # 'theta_op'  or  'N'

N_FIXED    = 50             # matrix size used when ANIMATE_OVER = 'theta_op'
N_MIN      = 2              # used when ANIMATE_OVER = 'N'
N_MAX      = 100

OP_FRAMES  = 120            # frames for one full 2π spectral rotation
FPS        = 24
OUT        = "polar_unwrap_spectral_rotation.mp4"

R_MIN      = 0.05
R_MAX      = 1.30
R_POINTS   = 420
PHI_POINTS = 720            # resolution in arg(z) direction

R_CRIT = np.exp(-0.5)

FIGSIZE = (12, 7)
DPI     = 160

# ── GCD Σ_N operator ─────────────────────────────────────────────────────────

def build_gcd_sigma(N):
    idx    = np.arange(1, N + 1)
    ii, jj = np.meshgrid(idx, idx, indexing='ij')
    G      = np.frompyfunc(gcd, 2, 1)(ii, jj).astype(float)
    return G / np.sqrt(ii * jj)

def get_eigen_cache(N):
    """Return (evals, log_ev, log_outer) for a given N."""
    Sigma      = build_gcd_sigma(N)
    evals      = np.sort(np.linalg.eigvalsh(Sigma))[::-1]
    evals      = np.maximum(evals, 1e-300)
    log_ev     = np.log(evals)
    ks         = np.arange(1, N + 1, dtype=float)
    log_outer  = np.outer(log_ev, ks)   # log_outer[j, m-1] = log(λ_j) * m
    return evals, log_ev, log_outer

# ── Newton's identities + Horner polynomial eval (exact copy from spectral.py) ─

def elementary_cx(power_sums):
    """Power sums → elementary symmetric polynomials over C (Newton's identities)."""
    n    = len(power_sums)
    e    = np.zeros(n + 1, dtype=complex)
    e[0] = 1.0 + 0j
    for m in range(1, n + 1):
        s = 0j
        for i in range(1, m + 1):
            s += ((-1) ** (i - 1)) * e[m - i] * power_sums[i - 1]
        e[m] = s / m
    return e

def poly_logdet(Zgrid, e, k, eps=1e-300):
    """
    Evaluate log det(I - z Σ^s) over the grid via Horner's method.
    Coefficients of det(I - z A) = Σ_m (-1)^m e_m z^m.
    Returns log of the polynomial value (complex).
    """
    val = np.zeros_like(Zgrid, dtype=complex)
    for j in range(k, -1, -1):
        val = val * Zgrid + ((-1) ** j) * e[j]
    val = np.where(np.abs(val) < eps, eps + 0j, val)
    return np.log(val)

# ── Polar z-grid ──────────────────────────────────────────────────────────────

phi      = np.linspace(0, 2 * np.pi, PHI_POINTS, endpoint=False)
r        = np.linspace(R_MIN, R_MAX, R_POINTS)
Phi, R_g = np.meshgrid(phi, r)
Z_grid   = R_g * np.exp(1j * Phi)    # shape (R_POINTS, PHI_POINTS)

# ── Frame computation ─────────────────────────────────────────────────────────

def compute_frame(theta_op, evals, log_ev, log_outer, N):
    """
    Compute phase field of det(I - z Σ_N^{exp(i θ_op)}) on the polar z-grid.

    Follows spectral.py compute_field exactly:
        s     = exp(i θ_op)
        p_k   = Tr(Σ^s)^k  via eigenvalues  [power sums of Σ^s]
        e_k   = elementary symmetric polys   [Newton's identities]
        zeros = λ_j^{-s}                     [zeros of the Fredholm det in z]
    """
    s   = np.exp(1j * theta_op)
    p_k = np.exp(log_outer * s).sum(axis=0)      # (N,)  power sums of Σ^s
    e_k = elementary_cx(p_k)

    logdet = poly_logdet(Z_grid, e_k, N)
    phase  = np.angle(np.exp(1j * np.imag(logdet)))
    logabs = np.real(logdet)

    # Zeros: z_j(θ_op) = λ_j^{-exp(i θ_op)} — same formula as spectral.py
    zzeros = np.exp(-s * log_ev)

    # Winding across the critical radius
    phase_uw  = np.unwrap(phase, axis=1)
    winding   = (phase_uw[:, -1] - phase_uw[:, 0]) / (2 * np.pi)
    i_below   = np.argmin(np.abs(r - R_CRIT * 0.985))
    i_above   = np.argmin(np.abs(r - R_CRIT * 1.015))
    delta_W   = winding[i_above] - winding[i_below]

    return phase, logabs, zzeros, delta_W

# ── Precompute eigendata ───────────────────────────────────────────────────────

if ANIMATE_OVER == 'theta_op':
    print(f"Computing N={N_FIXED} eigendecomposition …")
    evals0, log_ev0, log_outer0 = get_eigen_cache(N_FIXED)
    frames_list = list(np.linspace(0, 2 * np.pi, OP_FRAMES, endpoint=False))

else:  # 'N'
    print(f"Precomputing eigendecompositions N={N_MIN}→{N_MAX} …")
    eigen_cache = {}
    for _N in range(N_MIN, N_MAX + 1):
        eigen_cache[_N] = get_eigen_cache(_N)
    print("Done.")
    frames_list = list(range(N_MIN, N_MAX + 1))

# ── Figure ────────────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
fig.patch.set_facecolor("black")
ax.set_facecolor("black")

# Initial frame
if ANIMATE_OVER == 'theta_op':
    phase0, _, zzeros0, _ = compute_frame(0.0, evals0, log_ev0, log_outer0, N_FIXED)
else:
    ev0, lev0, lo0 = eigen_cache[N_MIN]
    phase0, _, zzeros0, _ = compute_frame(0.0, ev0, lev0, lo0, N_MIN)

im = ax.imshow(
    phase0,
    extent=[0, 2 * np.pi, R_MIN, R_MAX],
    origin='lower',
    aspect='auto',
    cmap='twilight',
    vmin=-np.pi, vmax=np.pi,
    interpolation='nearest',
)

# Reference circles
ax.axhline(R_CRIT, linestyle='--', linewidth=1.5, color='#00ee77')
ax.axhline(1.0,    linestyle=':',  linewidth=1.0, color='#ff9900', alpha=0.75)

# Zero dots in (arg z_j, |z_j|) space
phi_z0 = np.angle(zzeros0) % (2 * np.pi)
r_z0   = np.abs(zzeros0)
mask0  = (r_z0 >= R_MIN) & (r_z0 <= R_MAX)
zero_scat = ax.scatter(
    phi_z0[mask0], r_z0[mask0],
    c='white', s=22, alpha=0.9, zorder=5,
    linewidths=0.4, edgecolors='white',
)

# Labels
ax.text(2*np.pi * 0.985, R_CRIT + 0.014, r"$r = e^{-1/2}$",
        color='#00ee77', ha='right', va='bottom', fontsize=10)
ax.text(2*np.pi * 0.985, 1.0 + 0.014, r"$r = 1$",
        color='#ff9900', ha='right', va='bottom', fontsize=10)

title = ax.set_title('', color='white', fontsize=13, pad=10)

ax.set_xlabel(r"$\varphi = \arg(z)$ in $z = r\,e^{i\varphi}$", color='white')
ax.set_ylabel(r"$r = |z|$", color='white')
ax.set_xticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi])
ax.set_xticklabels([r'$0$', r'$\pi/2$', r'$\pi$', r'$3\pi/2$', r'$2\pi$'], color='white')
ax.tick_params(colors='white')
for spine in ax.spines.values():
    spine.set_color('white')

cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.025)
cbar.set_label(
    r"$\arg\det\!\left(I - z\,\Sigma_N^{\,e^{i\theta_{\mathrm{op}}}}\right)$",
    color='white',
)
cbar.ax.yaxis.set_tick_params(color='white')
plt.setp(cbar.ax.get_yticklabels(), color='white')

# Landmark annotations for spectral rotation
_LANDMARKS = {
    0:   "zeros on real axis: $z_j = 1/\\lambda_j$",
    90:  "all zeros on unit circle  $\\leftarrow$  exact phase-lock",
    180: "zeros at eigenvalues: $z_j = \\lambda_j$",
    270: "all zeros on unit circle  $\\leftarrow$  exact phase-lock",
}

# ── Update function ───────────────────────────────────────────────────────────

def update(frame):
    if ANIMATE_OVER == 'theta_op':
        theta_op = frame
        phase, _, zzeros, delta_W = compute_frame(
            theta_op, evals0, log_ev0, log_outer0, N_FIXED)
        deg = np.degrees(theta_op) % 360
        tag = next((v for k, v in _LANDMARKS.items() if abs(deg - k) < 1.6), '')
        label = (rf"$N={N_FIXED}$ · "
                 rf"$\theta_{{\mathrm{{op}}}}={deg:.0f}°$ · "
                 rf"$\det(I-z\,\Sigma^{{e^{{i\theta}}}})$"
                 + (f"  ·  {tag}" if tag else '')
                 + rf"  ·  $\Delta W\approx{delta_W:.2f}$")
    else:
        N = frame
        ev, lev, lo = eigen_cache[N]
        phase, _, zzeros, delta_W = compute_frame(0.0, ev, lev, lo, N)
        label = (rf"$\theta_{{\mathrm{{op}}}}=0°$ (plain det) · "
                 rf"$N={N}$ · "
                 rf"$\Delta W\approx{delta_W:.2f}$")

    im.set_data(phase)
    title.set_text(label)

    phi_z = np.angle(zzeros) % (2 * np.pi)
    r_z   = np.abs(zzeros)
    mask  = (r_z >= R_MIN) & (r_z <= R_MAX)
    zero_scat.set_offsets(
        np.column_stack([phi_z[mask], r_z[mask]]) if mask.any()
        else np.empty((0, 2))
    )

    return [im, zero_scat, title]

# ── Build and save ────────────────────────────────────────────────────────────

print("Rendering …")
ani = animation.FuncAnimation(
    fig, update, frames=frames_list,
    interval=1000 / FPS, blit=False,
)

ani.save(OUT, writer='ffmpeg', fps=FPS, dpi=DPI, bitrate=5000)
plt.close(fig)
print(f"Saved → {OUT}")

HTML(f'<video width="900" controls><source src="{OUT}" type="video/mp4"></video>')

In [ ]:
# ============================================================
# @title Polar unwrap animation — BRIGHTNESS version
# det(I - z Σ_N^{exp(i θ_op)})
#
# Visual:
#   x-axis = arg(z) = φ in z = r exp(i φ)
#   y-axis = |z| = r
#   color  = -log |det(I - z Σ_N^{exp(i θ_op)})|
#   dots   = zeros z_j(θ_op) = λ_j^{-exp(i θ_op)}
#
# Modes:
#   ANIMATE_OVER = 'theta_op'  -> spectral rotation at fixed N
#   ANIMATE_OVER = 'N'         -> grow N at fixed θ_op = 0
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from math import gcd
from IPython.display import HTML

# ── Config ───────────────────────────────────────────────────

ANIMATE_OVER = 'theta_op'   # 'theta_op' or 'N'

N_FIXED    = 50
N_MIN      = 2
N_MAX      = 100

OP_FRAMES  = 120
FPS        = 24
OUT        = "polar_unwrap_brightness_actual_object.mp4"

R_MIN      = 0.05
R_MAX      = 1.30
R_POINTS   = 420
PHI_POINTS = 720

R_CRIT = np.exp(-0.5)

# brightness clipping
BRIGHT_MIN = 0.0
BRIGHT_MAX = 10.0

FIGSIZE = (12, 7)
DPI     = 160

# ── GCD Σ_N operator ────────────────────────────────────────

def build_gcd_sigma(N):
    idx    = np.arange(1, N + 1)
    ii, jj = np.meshgrid(idx, idx, indexing='ij')
    G      = np.frompyfunc(gcd, 2, 1)(ii, jj).astype(float)
    return G / np.sqrt(ii * jj)

def get_eigen_cache(N):
    Sigma      = build_gcd_sigma(N)
    evals      = np.sort(np.linalg.eigvalsh(Sigma))[::-1]
    evals      = np.maximum(evals, 1e-300)
    log_ev     = np.log(evals)
    ks         = np.arange(1, N + 1, dtype=float)
    log_outer  = np.outer(log_ev, ks)
    return evals, log_ev, log_outer

# ── Newton identities + Horner (same as spectral.py) ───────

def elementary_cx(power_sums):
    n    = len(power_sums)
    e    = np.zeros(n + 1, dtype=complex)
    e[0] = 1.0 + 0j
    for m in range(1, n + 1):
        s = 0j
        for i in range(1, m + 1):
            s += ((-1) ** (i - 1)) * e[m - i] * power_sums[i - 1]
        e[m] = s / m
    return e

def poly_logdet(Zgrid, e, k, eps=1e-300):
    val = np.zeros_like(Zgrid, dtype=complex)
    for j in range(k, -1, -1):
        val = val * Zgrid + ((-1) ** j) * e[j]
    val = np.where(np.abs(val) < eps, eps + 0j, val)
    return np.log(val)

# ── Polar z-grid ────────────────────────────────────────────

phi      = np.linspace(0, 2 * np.pi, PHI_POINTS, endpoint=False)
r        = np.linspace(R_MIN, R_MAX, R_POINTS)
Phi, R_g = np.meshgrid(phi, r)
Z_grid   = R_g * np.exp(1j * Phi)

# ── Frame computation ───────────────────────────────────────

def compute_frame(theta_op, evals, log_ev, log_outer, N):
    s   = np.exp(1j * theta_op)
    p_k = np.exp(log_outer * s).sum(axis=0)
    e_k = elementary_cx(p_k)

    logdet = poly_logdet(Z_grid, e_k, N)
    logabs = np.real(logdet)
    bright = np.clip(-logabs, BRIGHT_MIN, BRIGHT_MAX)

    zzeros = np.exp(-s * log_ev)

    idx_crit = np.argmin(np.abs(r - R_CRIT))
    crit_band = bright[max(0, idx_crit - 2):min(len(r), idx_crit + 3), :]
    mean_crit_brightness = float(np.mean(crit_band))

    return bright, zzeros, mean_crit_brightness

# ── Precompute eigendata ────────────────────────────────────

if ANIMATE_OVER == 'theta_op':
    print(f"Computing N={N_FIXED} eigendecomposition …")
    evals0, log_ev0, log_outer0 = get_eigen_cache(N_FIXED)
    frames_list = list(np.linspace(0, 2 * np.pi, OP_FRAMES, endpoint=False))
else:
    print(f"Precomputing eigendecompositions N={N_MIN}→{N_MAX} …")
    eigen_cache = {}
    for _N in range(N_MIN, N_MAX + 1):
        eigen_cache[_N] = get_eigen_cache(_N)
    print("Done.")
    frames_list = list(range(N_MIN, N_MAX + 1))

# ── Figure ──────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
fig.patch.set_facecolor("black")
ax.set_facecolor("black")

if ANIMATE_OVER == 'theta_op':
    bright0, zzeros0, _ = compute_frame(0.0, evals0, log_ev0, log_outer0, N_FIXED)
else:
    ev0, lev0, lo0 = eigen_cache[N_MIN]
    bright0, zzeros0, _ = compute_frame(0.0, ev0, lev0, lo0, N_MIN)

im = ax.imshow(
    bright0,
    extent=[0, 2 * np.pi, R_MIN, R_MAX],
    origin='lower',
    aspect='auto',
    cmap='inferno',
    vmin=BRIGHT_MIN, vmax=BRIGHT_MAX,
    interpolation='nearest',
)

ax.axhline(R_CRIT, linestyle='--', linewidth=1.5, color='#00ee77')
ax.axhline(1.0,    linestyle=':',  linewidth=1.0, color='#ff9900', alpha=0.75)

phi_z0 = np.angle(zzeros0) % (2 * np.pi)
r_z0   = np.abs(zzeros0)
mask0  = (r_z0 >= R_MIN) & (r_z0 <= R_MAX)
zero_scat = ax.scatter(
    phi_z0[mask0], r_z0[mask0],
    c='cyan', s=18, alpha=0.9, zorder=5,
    linewidths=0.3, edgecolors='white',
)

ax.text(2*np.pi * 0.985, R_CRIT + 0.014, r"$r = e^{-1/2}$",
        color='#00ee77', ha='right', va='bottom', fontsize=10)
ax.text(2*np.pi * 0.985, 1.0 + 0.014, r"$r = 1$",
        color='#ff9900', ha='right', va='bottom', fontsize=10)

title = ax.set_title('', color='white', fontsize=13, pad=10)

ax.set_xlabel(r"$\varphi = \arg(z)$ in $z = r\,e^{i\varphi}$", color='white')
ax.set_ylabel(r"$r = |z|$", color='white')
ax.set_xticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi])
ax.set_xticklabels([r'$0$', r'$\pi/2$', r'$\pi$', r'$3\pi/2$', r'$2\pi$'], color='white')
ax.tick_params(colors='white')
for spine in ax.spines.values():
    spine.set_color('white')

cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.025)
cbar.set_label(
    r"$-\log\left|\det\!\left(I - z\,\Sigma_N^{\,e^{i\theta_{\mathrm{op}}}}\right)\right|$",
    color='white',
)
cbar.ax.yaxis.set_tick_params(color='white')
plt.setp(cbar.ax.get_yticklabels(), color='white')

_LANDMARKS = {
    0:   r"$\theta_{\rm op}=0^\circ$: zeros at $z_j=1/\lambda_j$",
    90:  r"$\theta_{\rm op}=90^\circ$: all zeros on $|z|=1$",
    180: r"$\theta_{\rm op}=180^\circ$: zeros at $z_j=\lambda_j$",
    270: r"$\theta_{\rm op}=270^\circ$: all zeros on $|z|=1$",
}

# ── Update ──────────────────────────────────────────────────

def update(frame):
    if ANIMATE_OVER == 'theta_op':
        theta_op = frame
        bright, zzeros, crit_mean = compute_frame(theta_op, evals0, log_ev0, log_outer0, N_FIXED)

        deg = int(round(np.degrees(theta_op))) % 360
        landmark = ""
        for k, txt in _LANDMARKS.items():
            if abs(deg - k) <= 2:
                landmark = "  ·  " + txt
                break

        title.set_text(
            rf"brightness unwrap   |   $N={N_FIXED}$   |   "
            rf"$\theta_{{\rm op}}={deg}^\circ$   |   "
            rf"mean near $e^{{-1/2}}$ = {crit_mean:.2f}{landmark}"
        )

    else:
        N = frame
        ev, lev, lo = eigen_cache[N]
        theta_op = 0.0
        bright, zzeros, crit_mean = compute_frame(theta_op, ev, lev, lo, N)

        title.set_text(
            rf"brightness unwrap   |   $N={N}$   |   "
            rf"$\theta_{{\rm op}}=0^\circ$   |   "
            rf"mean near $e^{{-1/2}}$ = {crit_mean:.2f}"
        )

    im.set_data(bright)

    phi_z = np.angle(zzeros) % (2 * np.pi)
    r_z   = np.abs(zzeros)
    mask  = (r_z >= R_MIN) & (r_z <= R_MAX)
    zero_scat.set_offsets(np.column_stack([phi_z[mask], r_z[mask]]))

    return [im, zero_scat, title]

ani = animation.FuncAnimation(
    fig, update, frames=frames_list, interval=1000/FPS, blit=False
)

ani.save(OUT, writer="ffmpeg", fps=FPS, dpi=DPI, bitrate=5000)

plt.close(fig)

print(f"Saved -> {OUT}")
HTML(f"""
<video width="900" controls>
  <source src="{OUT}" type="video/mp4">
</video>
""")

In [ ]:
# ============================================================
# @title Fixed N=100 — BRIGHTNESS unwrap with moving radius scan line
# for the actual object:
#   det(I - z Σ_N^{exp(i θ_op)})
#
# Background:
#   color = -log |det(I - z Σ_N^{exp(i θ_op)})|
#
# Animation:
#   a horizontal scan line moves upward through r
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from math import gcd
from IPython.display import HTML

# ── Config ───────────────────────────────────────────────────

N_FIXED         = 100
THETA_OP_FIXED  = 0.0      # try 0.0, np.pi/2, np.pi, 3*np.pi/2

R_MIN           = 0.05
R_MAX           = 1.30
R_POINTS        = 420
PHI_POINTS      = 720

FPS             = 24
OUT             = "polar_unwrap_brightness_scanline_actual_object_N100.mp4"

R_CRIT          = np.exp(-0.5)

BRIGHT_MIN      = 0.0
BRIGHT_MAX      = 10.0

FIGSIZE         = (12, 7)
DPI             = 160

# ── GCD Σ_N operator ────────────────────────────────────────

def build_gcd_sigma(N):
    idx    = np.arange(1, N + 1)
    ii, jj = np.meshgrid(idx, idx, indexing='ij')
    G      = np.frompyfunc(gcd, 2, 1)(ii, jj).astype(float)
    return G / np.sqrt(ii * jj)

def get_eigen_cache(N):
    Sigma      = build_gcd_sigma(N)
    evals      = np.sort(np.linalg.eigvalsh(Sigma))[::-1]
    evals      = np.maximum(evals, 1e-300)
    log_ev     = np.log(evals)
    ks         = np.arange(1, N + 1, dtype=float)
    log_outer  = np.outer(log_ev, ks)
    return evals, log_ev, log_outer

# ── Newton identities + Horner ──────────────────────────────

def elementary_cx(power_sums):
    n    = len(power_sums)
    e    = np.zeros(n + 1, dtype=complex)
    e[0] = 1.0 + 0j
    for m in range(1, n + 1):
        s = 0j
        for i in range(1, m + 1):
            s += ((-1) ** (i - 1)) * e[m - i] * power_sums[i - 1]
        e[m] = s / m
    return e

def poly_logdet(Zgrid, e, k, eps=1e-300):
    val = np.zeros_like(Zgrid, dtype=complex)
    for j in range(k, -1, -1):
        val = val * Zgrid + ((-1) ** j) * e[j]
    val = np.where(np.abs(val) < eps, eps + 0j, val)
    return np.log(val)

# ── Polar z-grid ────────────────────────────────────────────

phi      = np.linspace(0, 2 * np.pi, PHI_POINTS, endpoint=False)
r        = np.linspace(R_MIN, R_MAX, R_POINTS)
Phi, R_g = np.meshgrid(phi, r)
Z_grid   = R_g * np.exp(1j * Phi)

# ── Compute fixed field ─────────────────────────────────────

evals, log_ev, log_outer = get_eigen_cache(N_FIXED)

s   = np.exp(1j * THETA_OP_FIXED)
p_k = np.exp(log_outer * s).sum(axis=0)
e_k = elementary_cx(p_k)

logdet = poly_logdet(Z_grid, e_k, N_FIXED)
logabs = np.real(logdet)
bright = np.clip(-logabs, BRIGHT_MIN, BRIGHT_MAX)

zzeros = np.exp(-s * log_ev)
phi_z  = np.angle(zzeros) % (2 * np.pi)
r_z    = np.abs(zzeros)
mask_z = (r_z >= R_MIN) & (r_z <= R_MAX)

# ── Figure ──────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
fig.patch.set_facecolor("black")
ax.set_facecolor("black")

im = ax.imshow(
    bright,
    extent=[0, 2 * np.pi, R_MIN, R_MAX],
    origin='lower',
    aspect='auto',
    cmap='inferno',
    vmin=BRIGHT_MIN, vmax=BRIGHT_MAX,
    interpolation='nearest',
)

ax.axhline(R_CRIT, linestyle='--', linewidth=1.5, color='#00ee77')
ax.axhline(1.0,    linestyle=':',  linewidth=1.0, color='#ff9900', alpha=0.75)

ax.scatter(
    phi_z[mask_z], r_z[mask_z],
    c='cyan', s=18, alpha=0.9, zorder=5,
    linewidths=0.3, edgecolors='white',
)

scan_line = ax.axhline(R_MIN, linewidth=2.2, color='magenta', zorder=6)

ax.text(2*np.pi * 0.985, R_CRIT + 0.014, r"$r = e^{-1/2}$",
        color='#00ee77', ha='right', va='bottom', fontsize=10)
ax.text(2*np.pi * 0.985, 1.0 + 0.014, r"$r = 1$",
        color='#ff9900', ha='right', va='bottom', fontsize=10)

title = ax.set_title('', color='white', fontsize=13, pad=10)

ax.set_xlabel(r"$\varphi = \arg(z)$ in $z = r\,e^{i\varphi}$", color='white')
ax.set_ylabel(r"$r = |z|$", color='white')
ax.set_xticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi])
ax.set_xticklabels([r'$0$', r'$\pi/2$', r'$\pi$', r'$3\pi/2$', r'$2\pi$'], color='white')
ax.tick_params(colors='white')
for spine in ax.spines.values():
    spine.set_color('white')

cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.025)
cbar.set_label(
    r"$-\log\left|\det\!\left(I - z\,\Sigma_{100}^{\,e^{i\theta_{\mathrm{op}}}}\right)\right|$",
    color='white',
)
cbar.ax.yaxis.set_tick_params(color='white')
plt.setp(cbar.ax.get_yticklabels(), color='white')

# ── Animation ───────────────────────────────────────────────

scan_r_values = np.linspace(R_MIN, R_MAX, 240)

def update(frame_idx):
    rr = scan_r_values[frame_idx]
    scan_line.set_ydata([rr, rr])

    idx = np.argmin(np.abs(r - rr))
    slice_mean = float(np.mean(bright[idx, :]))
    slice_max  = float(np.max(bright[idx, :]))

    deg = int(round(np.degrees(THETA_OP_FIXED))) % 360

    title.set_text(
        rf"fixed $N=100$   |   "
        rf"$\theta_{{\rm op}}={deg}^\circ$   |   "
        rf"scan radius $r={rr:.4f}$   |   "
        rf"mean = {slice_mean:.2f}   |   max = {slice_max:.2f}"
    )

    return [scan_line, title]

ani = animation.FuncAnimation(
    fig, update, frames=len(scan_r_values), interval=1000/FPS, blit=False
)

ani.save(OUT, writer="ffmpeg", fps=FPS, dpi=DPI, bitrate=5000)

plt.close(fig)

print(f"Saved -> {OUT}")
HTML(f"""
<video width="900" controls>
  <source src="{OUT}" type="video/mp4">
</video>
""")

In [ ]:
# ============================================================
# @title 4-panel fixed N=100 brightness unwrap
# Actual object:
#   det(I - z Σ_N^{exp(i θ_op)})
#
# Panels:
#   θ_op = 0°, 90°, 180°, 270°
#
# Visual:
#   x-axis = φ = arg(z)
#   y-axis = r = |z|
#   color  = -log |det(I - z Σ_N^{exp(i θ_op)})|
#   dots   = zeros z_j(θ_op) = λ_j^{-exp(i θ_op)}
#
# Animation:
#   A horizontal scanline moves upward through r
#   across all four panels simultaneously.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from math import gcd
from IPython.display import HTML

# ── Config ───────────────────────────────────────────────────

N_FIXED = 100

THETA_OP_LIST = [
    0.0,
    np.pi / 2,
    np.pi,
    3 * np.pi / 2,
]

THETA_LABELS = [
    r"$\theta_{\rm op}=0^\circ$   zeros: $z_j=1/\lambda_j$",
    r"$\theta_{\rm op}=90^\circ$   zeros locked on $|z|=1$",
    r"$\theta_{\rm op}=180^\circ$   zeros: $z_j=\lambda_j$",
    r"$\theta_{\rm op}=270^\circ$   zeros locked on $|z|=1$",
]

R_MIN      = 0.05
R_MAX      = 1.30
R_POINTS   = 420
PHI_POINTS = 720

R_CRIT = np.exp(-0.5)

BRIGHT_MIN = 0.0
BRIGHT_MAX = 10.0

FPS = 24
OUT = "four_panel_brightness_scanline_actual_object_N100.mp4"

FIGSIZE = (16, 10)
DPI = 150

# ── GCD Σ_N operator ────────────────────────────────────────

def build_gcd_sigma(N):
    idx = np.arange(1, N + 1)
    ii, jj = np.meshgrid(idx, idx, indexing="ij")
    G = np.frompyfunc(gcd, 2, 1)(ii, jj).astype(float)
    return G / np.sqrt(ii * jj)

def get_eigen_cache(N):
    Sigma = build_gcd_sigma(N)
    evals = np.sort(np.linalg.eigvalsh(Sigma))[::-1]
    evals = np.maximum(evals, 1e-300)
    log_ev = np.log(evals)
    ks = np.arange(1, N + 1, dtype=float)
    log_outer = np.outer(log_ev, ks)
    return evals, log_ev, log_outer

# ── Newton identities + Horner polynomial eval ──────────────

def elementary_cx(power_sums):
    n = len(power_sums)
    e = np.zeros(n + 1, dtype=complex)
    e[0] = 1.0 + 0j

    for m in range(1, n + 1):
        s = 0j
        for i in range(1, m + 1):
            s += ((-1) ** (i - 1)) * e[m - i] * power_sums[i - 1]
        e[m] = s / m

    return e

def poly_logdet(Zgrid, e, k, eps=1e-300):
    val = np.zeros_like(Zgrid, dtype=complex)

    for j in range(k, -1, -1):
        val = val * Zgrid + ((-1) ** j) * e[j]

    val = np.where(np.abs(val) < eps, eps + 0j, val)
    return np.log(val)

# ── Polar z-grid ────────────────────────────────────────────

phi = np.linspace(0, 2 * np.pi, PHI_POINTS, endpoint=False)
r = np.linspace(R_MIN, R_MAX, R_POINTS)

Phi, R_g = np.meshgrid(phi, r)
Z_grid = R_g * np.exp(1j * Phi)

# ── Field computation ───────────────────────────────────────

def compute_brightness_panel(theta_op, evals, log_ev, log_outer, N):
    """
    Compute brightness field for:
        det(I - z Σ_N^{exp(i theta_op)})

    Uses:
        p_k = Tr((Σ^s)^k), s = exp(i theta_op)
        elementary symmetric polys via Newton identities
        polynomial det via Horner
    """
    s = np.exp(1j * theta_op)

    p_k = np.exp(log_outer * s).sum(axis=0)
    e_k = elementary_cx(p_k)

    logdet = poly_logdet(Z_grid, e_k, N)
    logabs = np.real(logdet)

    bright = np.clip(-logabs, BRIGHT_MIN, BRIGHT_MAX)

    zzeros = np.exp(-s * log_ev)

    return bright, zzeros

# ── Compute all four fixed panels ───────────────────────────

print(f"Computing eigendecomposition for N={N_FIXED}...")
evals, log_ev, log_outer = get_eigen_cache(N_FIXED)

panel_data = []

print("Computing four θ_op panels...")
for theta_op in THETA_OP_LIST:
    bright, zzeros = compute_brightness_panel(
        theta_op, evals, log_ev, log_outer, N_FIXED
    )
    panel_data.append((bright, zzeros))

print("Done.")

# ── Figure ──────────────────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=FIGSIZE, dpi=DPI, sharex=True, sharey=True)
fig.patch.set_facecolor("black")
axes = axes.ravel()

images = []
scan_lines = []

for ax, label, (bright, zzeros) in zip(axes, THETA_LABELS, panel_data):
    ax.set_facecolor("black")

    im = ax.imshow(
        bright,
        extent=[0, 2 * np.pi, R_MIN, R_MAX],
        origin="lower",
        aspect="auto",
        cmap="inferno",
        vmin=BRIGHT_MIN,
        vmax=BRIGHT_MAX,
        interpolation="nearest",
    )
    images.append(im)

    # reference radii
    ax.axhline(R_CRIT, linestyle="--", linewidth=1.4, color="#00ee77")
    ax.axhline(1.0, linestyle=":", linewidth=1.0, color="#ff9900", alpha=0.75)

    # zeros in polar unwrap coordinates
    phi_z = np.angle(zzeros) % (2 * np.pi)
    r_z = np.abs(zzeros)
    mask = (r_z >= R_MIN) & (r_z <= R_MAX)

    ax.scatter(
        phi_z[mask],
        r_z[mask],
        c="cyan",
        s=14,
        alpha=0.85,
        zorder=5,
        linewidths=0.25,
        edgecolors="white",
    )

    scan = ax.axhline(R_MIN, linewidth=2.0, color="magenta", zorder=6)
    scan_lines.append(scan)

    ax.set_title(label, color="white", fontsize=11, pad=8)

    ax.text(
        2 * np.pi * 0.985,
        R_CRIT + 0.012,
        r"$e^{-1/2}$",
        color="#00ee77",
        ha="right",
        va="bottom",
        fontsize=9,
    )

    ax.text(
        2 * np.pi * 0.985,
        1.0 + 0.012,
        r"$1$",
        color="#ff9900",
        ha="right",
        va="bottom",
        fontsize=9,
    )

    ax.set_xticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi])
    ax.set_xticklabels(
        [r"$0$", r"$\pi/2$", r"$\pi$", r"$3\pi/2$", r"$2\pi$"],
        color="white",
    )

    ax.tick_params(colors="white")

    for spine in ax.spines.values():
        spine.set_color("white")

axes[0].set_ylabel(r"$r=|z|$", color="white")
axes[2].set_ylabel(r"$r=|z|$", color="white")
axes[2].set_xlabel(r"$\varphi=\arg(z)$", color="white")
axes[3].set_xlabel(r"$\varphi=\arg(z)$", color="white")

suptitle = fig.suptitle(
    "",
    color="white",
    fontsize=14,
    y=0.965,
)

# one shared colorbar
cbar = fig.colorbar(images[0], ax=axes, fraction=0.035, pad=0.025)
cbar.set_label(
    r"$-\log\left|\det\!\left(I-z\Sigma_{100}^{e^{i\theta_{\rm op}}}\right)\right|$",
    color="white",
)
cbar.ax.yaxis.set_tick_params(color="white")
plt.setp(cbar.ax.get_yticklabels(), color="white")

# ── Animation ───────────────────────────────────────────────

scan_r_values = np.linspace(R_MIN, R_MAX, 240)

def update(frame_idx):
    rr = scan_r_values[frame_idx]

    for scan in scan_lines:
        scan.set_ydata([rr, rr])

    # Diagnostic values at scanline for each panel
    diagnostics = []
    idx = np.argmin(np.abs(r - rr))

    for theta_op, (bright, _) in zip(THETA_OP_LIST, panel_data):
        deg = int(round(np.degrees(theta_op))) % 360
        slice_mean = float(np.mean(bright[idx, :]))
        slice_max = float(np.max(bright[idx, :]))
        diagnostics.append(f"{deg}° mean={slice_mean:.2f}, max={slice_max:.2f}")

    suptitle.set_text(
        rf"fixed $N=100$ brightness unwrap   |   scan radius $r={rr:.4f}$   |   "
        + "   ·   ".join(diagnostics)
    )

    return [*scan_lines, suptitle]

ani = animation.FuncAnimation(
    fig,
    update,
    frames=len(scan_r_values),
    interval=1000 / FPS,
    blit=False,
)

ani.save(
    OUT,
    writer="ffmpeg",
    fps=FPS,
    dpi=DPI,
    bitrate=6000,
)

plt.close(fig)

print(f"Saved -> {OUT}")
HTML(f"""
<video width="1000" controls>
  <source src="{OUT}" type="video/mp4">
</video>
""")

In [ ]:
# ============================================================
# @title Branch-cut vortex microscope
# Actual object:
#   det(I - z Σ_N^{exp(i θ_op)})
#
# Purpose:
#   Visualize the unresolved "dust" near φ=0/2π as individual
#   phase vortices near the polar branch cut.
#
# Visual:
#   x-axis = φ_cut, centered around the positive real axis
#            φ_cut ∈ [-PHI_WINDOW, +PHI_WINDOW]
#   y-axis = r = |z|
#   color  = arg det(I - z Σ_N^{exp(i θ_op)})
#   dots   = exact zeros z_j(θ_op)=λ_j^{-exp(i θ_op)}
#   x marks = detected plaquette phase vortices
#
# Suggested:
#   θ_op near 270° to 285°.
#   Around 276° you should see the branch-cut vortex cluster.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from math import gcd
from IPython.display import HTML

# ── Config ───────────────────────────────────────────────────

N_FIXED = 50

THETA_START_DEG = 260
THETA_END_DEG   = 286
OP_FRAMES       = 90

FPS = 24
OUT = "branch_cut_vortex_microscope.mp4"

# Window around the branch cut φ=0.
# Old plot had φ=0 at left edge and φ=2π at right edge.
# This puts both sides together in one continuous local view.
PHI_WINDOW = 0.75          # radians; increase to see wider neighborhood
PHI_POINTS = 2160          # high angular resolution

R_MIN    = 0.50
R_MAX    = 1.05
R_POINTS = 720

R_CRIT = np.exp(-0.5)

FIGSIZE = (12, 7)
DPI = 160

# Vortex detector threshold.
# A true simple zero usually gives charge +1.
# Numerical / unresolved areas can produce fractional-looking noise,
# so we only plot cells close to integer winding.
VORTEX_THRESHOLD = 0.55

# ── GCD Σ_N operator ────────────────────────────────────────

def build_gcd_sigma(N):
    idx = np.arange(1, N + 1)
    ii, jj = np.meshgrid(idx, idx, indexing="ij")
    G = np.frompyfunc(gcd, 2, 1)(ii, jj).astype(float)
    return G / np.sqrt(ii * jj)

def get_eigen_cache(N):
    Sigma = build_gcd_sigma(N)
    evals = np.sort(np.linalg.eigvalsh(Sigma))[::-1]
    evals = np.maximum(evals, 1e-300)

    log_ev = np.log(evals)
    ks = np.arange(1, N + 1, dtype=float)
    log_outer = np.outer(log_ev, ks)

    return evals, log_ev, log_outer

# ── Newton identities + Horner polynomial eval ──────────────

def elementary_cx(power_sums):
    n = len(power_sums)
    e = np.zeros(n + 1, dtype=complex)
    e[0] = 1.0 + 0j

    for m in range(1, n + 1):
        s = 0j
        for i in range(1, m + 1):
            s += ((-1) ** (i - 1)) * e[m - i] * power_sums[i - 1]
        e[m] = s / m

    return e

def poly_logdet(Zgrid, e, k, eps=1e-300):
    val = np.zeros_like(Zgrid, dtype=complex)

    for j in range(k, -1, -1):
        val = val * Zgrid + ((-1) ** j) * e[j]

    val = np.where(np.abs(val) < eps, eps + 0j, val)
    return np.log(val)

# ── Phase helpers ───────────────────────────────────────────

def wrap_angle(x):
    """
    Wrap angle to (-π, π].
    Used for phase differences.
    """
    return (x + np.pi) % (2 * np.pi) - np.pi

def plaquette_vorticity(phase):
    """
    Compute integer-like phase winding around each grid cell.

    phase shape: (R_POINTS, PHI_POINTS)

    Returns:
        charge field shape: (R_POINTS - 1, PHI_POINTS - 1)

    A charge near +1 means a positive phase vortex / zero
    sits inside that cell.
    """
    p00 = phase[:-1, :-1]
    p10 = phase[1:, :-1]
    p11 = phase[1:, 1:]
    p01 = phase[:-1, 1:]

    d1 = wrap_angle(p10 - p00)
    d2 = wrap_angle(p11 - p10)
    d3 = wrap_angle(p01 - p11)
    d4 = wrap_angle(p00 - p01)

    return (d1 + d2 + d3 + d4) / (2 * np.pi)

# ── Branch-cut-centered polar grid ──────────────────────────

phi_cut = np.linspace(-PHI_WINDOW, PHI_WINDOW, PHI_POINTS)
r = np.linspace(R_MIN, R_MAX, R_POINTS)

Phi_cut, R_g = np.meshgrid(phi_cut, r)

# Physical z-angle is modulo 2π.
# Negative φ_cut values correspond to angles near 2π.
Phi_phys = np.mod(Phi_cut, 2 * np.pi)
Z_grid = R_g * np.exp(1j * Phi_phys)

# Centers of plaquette cells, used to plot detected vortices.
phi_centers = 0.5 * (phi_cut[:-1] + phi_cut[1:])
r_centers = 0.5 * (r[:-1] + r[1:])
Phi_c, R_c = np.meshgrid(phi_centers, r_centers)

# ── Frame computation ───────────────────────────────────────

def compute_frame(theta_op, evals, log_ev, log_outer, N):
    s = np.exp(1j * theta_op)

    p_k = np.exp(log_outer * s).sum(axis=0)
    e_k = elementary_cx(p_k)

    logdet = poly_logdet(Z_grid, e_k, N)
    phase = np.angle(np.exp(1j * np.imag(logdet)))

    zzeros = np.exp(-s * log_ev)

    # Exact zero positions in branch-cut coordinates.
    phi_z = np.angle(zzeros)
    phi_z = wrap_angle(phi_z)   # now centered around φ=0
    r_z = np.abs(zzeros)

    visible = (
        (phi_z >= -PHI_WINDOW) &
        (phi_z <=  PHI_WINDOW) &
        (r_z >= R_MIN) &
        (r_z <= R_MAX)
    )

    # Plaquette vortex detector.
    charge = plaquette_vorticity(phase)
    vortex_mask = np.abs(charge) >= VORTEX_THRESHOLD

    vortex_phi = Phi_c[vortex_mask]
    vortex_r = R_c[vortex_mask]
    vortex_q = charge[vortex_mask]

    return phase, phi_z[visible], r_z[visible], vortex_phi, vortex_r, vortex_q

# ── Precompute eigendata ────────────────────────────────────

print(f"Computing eigendecomposition for N={N_FIXED}...")
evals, log_ev, log_outer = get_eigen_cache(N_FIXED)

theta_frames = np.deg2rad(
    np.linspace(THETA_START_DEG, THETA_END_DEG, OP_FRAMES)
)

# ── Figure ──────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
fig.patch.set_facecolor("black")
ax.set_facecolor("black")

phase0, phi_z0, r_z0, vortex_phi0, vortex_r0, vortex_q0 = compute_frame(
    theta_frames[0], evals, log_ev, log_outer, N_FIXED
)

im = ax.imshow(
    phase0,
    extent=[-PHI_WINDOW, PHI_WINDOW, R_MIN, R_MAX],
    origin="lower",
    aspect="auto",
    cmap="twilight",
    vmin=-np.pi,
    vmax=np.pi,
    interpolation="nearest",
)

# Reference lines
ax.axhline(R_CRIT, linestyle="--", linewidth=1.5, color="#00ee77")
ax.axhline(1.0, linestyle=":", linewidth=1.1, color="#ff9900", alpha=0.8)
ax.axvline(0.0, linestyle="--", linewidth=1.0, color="white", alpha=0.45)

zero_scat = ax.scatter(
    phi_z0,
    r_z0,
    c="white",
    s=26,
    alpha=0.95,
    zorder=5,
    linewidths=0.4,
    edgecolors="black",
    label="exact zeros",
)

vortex_scat = ax.scatter(
    vortex_phi0,
    vortex_r0,
    c="cyan",
    s=34,
    marker="x",
    alpha=0.95,
    zorder=6,
    linewidths=1.1,
    label="detected phase vortices",
)

ax.text(
    PHI_WINDOW * 0.98,
    R_CRIT + 0.012,
    r"$r=e^{-1/2}$",
    color="#00ee77",
    ha="right",
    va="bottom",
    fontsize=10,
)

ax.text(
    PHI_WINDOW * 0.98,
    1.0 + 0.012,
    r"$r=1$",
    color="#ff9900",
    ha="right",
    va="bottom",
    fontsize=10,
)

title = ax.set_title("", color="white", fontsize=13, pad=10)

ax.set_xlabel(r"branch-cut angle $\varphi_{\rm cut}$ around positive real axis", color="white")
ax.set_ylabel(r"$r=|z|$", color="white")

ax.tick_params(colors="white")
for spine in ax.spines.values():
    spine.set_color("white")

leg = ax.legend(loc="lower right", facecolor="black", edgecolor="white")
for text in leg.get_texts():
    text.set_color("white")

cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.025)
cbar.set_label(
    r"$\arg\det(I-z\Sigma_N^{e^{i\theta_{\rm op}}})$",
    color="white",
)
cbar.ax.yaxis.set_tick_params(color="white")
plt.setp(cbar.ax.get_yticklabels(), color="white")

# ── Animation update ────────────────────────────────────────

def update(theta_op):
    phase, phi_z, r_z, vortex_phi, vortex_r, vortex_q = compute_frame(
        theta_op, evals, log_ev, log_outer, N_FIXED
    )

    im.set_data(phase)

    if len(phi_z):
        zero_scat.set_offsets(np.column_stack([phi_z, r_z]))
    else:
        zero_scat.set_offsets(np.empty((0, 2)))

    if len(vortex_phi):
        vortex_scat.set_offsets(np.column_stack([vortex_phi, vortex_r]))
    else:
        vortex_scat.set_offsets(np.empty((0, 2)))

    deg = np.degrees(theta_op) % 360

    title.set_text(
        rf"branch-cut vortex microscope   |   "
        rf"$N={N_FIXED}$   |   "
        rf"$\theta_{{\rm op}}={deg:.1f}^\circ$   |   "
        rf"visible zeros={len(phi_z)}   |   detected vortices={len(vortex_phi)}"
    )

    return [im, zero_scat, vortex_scat, title]

ani = animation.FuncAnimation(
    fig,
    update,
    frames=theta_frames,
    interval=1000 / FPS,
    blit=False,
)

ani.save(
    OUT,
    writer="ffmpeg",
    fps=FPS,
    dpi=DPI,
    bitrate=7000,
)

plt.close(fig)

print(f"Saved -> {OUT}")
HTML(f"""
<video width="900" controls>
  <source src="{OUT}" type="video/mp4">
</video>
""")

In [ ]:
# ============================================================
# @title Zero-trajectory resonance plot
#
# Actual zero rule from your object:
#   z_j(θ_op) = λ_j^{-exp(i θ_op)}
#
# Visual:
#   x-axis = φ_cut = arg(z_j), centered around positive real axis
#   y-axis = r_j = |z_j|
#
# Animation:
#   θ_op sweeps through a window near a locking angle.
#
# Purpose:
#   Show unit-circle locking, release/unlocking, and branch-cut bunching
#   using only the zero trajectories, no phase-field background.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from math import gcd
from IPython.display import HTML

# ── Config ───────────────────────────────────────────────────

N_FIXED = 50

# Sweep around the locking/release region.
# Good windows:
#   250 → 290 : around 270° lock/release
#    70 → 110 : around 90° lock/release
THETA_START_DEG = 250
THETA_END_DEG   = 290
FRAMES          = 180

FPS = 30
OUT = "zero_trajectory_resonance_plot.mp4"

# Branch-cut-centered angular window.
PHI_WINDOW = 1.05

# Radius window.
R_MIN = 0.45
R_MAX = 1.15

R_CRIT = np.exp(-0.5)

FIGSIZE = (12, 7)
DPI = 160

# Number of previous positions to leave as trails.
TRAIL_LENGTH = 42

# Optional: emphasize eigenvalues above/below 1
# λ_j > 1 and λ_j < 1 move to opposite radial sides near the quarter-turns.
COLOR_BY = "lambda_side"   # "lambda_side" or "index"

# ── GCD Σ_N operator ────────────────────────────────────────

def build_gcd_sigma(N):
    idx = np.arange(1, N + 1)
    ii, jj = np.meshgrid(idx, idx, indexing="ij")
    G = np.frompyfunc(gcd, 2, 1)(ii, jj).astype(float)
    return G / np.sqrt(ii * jj)

def wrap_angle(x):
    return (x + np.pi) % (2 * np.pi) - np.pi

# ── Eigenvalues and zero orbit ──────────────────────────────

Sigma = build_gcd_sigma(N_FIXED)
evals = np.sort(np.linalg.eigvalsh(Sigma))[::-1]
evals = np.maximum(evals, 1e-300)
log_ev = np.log(evals)

theta_values = np.deg2rad(np.linspace(THETA_START_DEG, THETA_END_DEG, FRAMES))

def zero_positions(theta_op):
    """
    z_j(θ) = exp(-exp(iθ) log λ_j)

    Returns:
        phi_cut, radius, complex z
    """
    s = np.exp(1j * theta_op)
    z = np.exp(-s * log_ev)

    phi = wrap_angle(np.angle(z))
    rad = np.abs(z)

    return phi, rad, z

# Precompute all positions for trails and density diagnostics.
all_phi = []
all_r = []

for th in theta_values:
    ph, rr, _ = zero_positions(th)
    all_phi.append(ph)
    all_r.append(rr)

all_phi = np.array(all_phi)   # shape (FRAMES, N_FIXED)
all_r = np.array(all_r)

# ── Figure ──────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
fig.patch.set_facecolor("black")
ax.set_facecolor("black")

ax.set_xlim(-PHI_WINDOW, PHI_WINDOW)
ax.set_ylim(R_MIN, R_MAX)

# Reference lines
ax.axhline(1.0, linestyle=":", linewidth=1.3, color="#ff9900", alpha=0.85)
ax.axhline(R_CRIT, linestyle="--", linewidth=1.3, color="#00ee77", alpha=0.85)
ax.axvline(0.0, linestyle="--", linewidth=1.0, color="white", alpha=0.35)

ax.text(
    PHI_WINDOW * 0.97,
    1.0 + 0.012,
    r"$r=1$ lock manifold",
    color="#ff9900",
    ha="right",
    va="bottom",
    fontsize=10,
)

ax.text(
    PHI_WINDOW * 0.97,
    R_CRIT + 0.012,
    r"$r=e^{-1/2}$",
    color="#00ee77",
    ha="right",
    va="bottom",
    fontsize=10,
)

# Faint full trajectory background over the chosen θ-window.
# This shows where the zero-orbit fan lives.
for j in range(N_FIXED):
    mask = (
        (all_phi[:, j] >= -PHI_WINDOW) &
        (all_phi[:, j] <=  PHI_WINDOW) &
        (all_r[:, j] >= R_MIN) &
        (all_r[:, j] <= R_MAX)
    )

    if np.any(mask):
        ax.plot(
            all_phi[mask, j],
            all_r[mask, j],
            linewidth=0.7,
            alpha=0.16,
            color="white",
        )

# Color setup for current points
if COLOR_BY == "lambda_side":
    # λ > 1: cyan-ish; λ < 1: magenta-ish
    point_colors = np.where(evals > 1.0, 1.0, 0.0)
    cmap = plt.get_cmap("cool")
else:
    point_colors = np.arange(N_FIXED)
    cmap = plt.get_cmap("viridis")

# Current zero points
phi0, r0, _ = zero_positions(theta_values[0])
visible0 = (
    (phi0 >= -PHI_WINDOW) &
    (phi0 <=  PHI_WINDOW) &
    (r0 >= R_MIN) &
    (r0 <= R_MAX)
)

scat = ax.scatter(
    phi0[visible0],
    r0[visible0],
    c=point_colors[visible0],
    cmap=cmap,
    s=44,
    alpha=0.95,
    edgecolors="white",
    linewidths=0.45,
    zorder=5,
)

# Trail points, updated frame by frame
trail_scat = ax.scatter(
    [],
    [],
    c=[],
    cmap=cmap,
    s=16,
    alpha=0.35,
    edgecolors="none",
    zorder=4,
)

title = ax.set_title("", color="white", fontsize=13, pad=10)

ax.set_xlabel(
    r"branch-cut angle $\varphi_{\rm cut}=\arg(z_j)$ around positive real axis",
    color="white",
)
ax.set_ylabel(r"zero radius $r_j=|z_j|$", color="white")

ax.tick_params(colors="white")
for spine in ax.spines.values():
    spine.set_color("white")

# Small explanatory annotation
ax.text(
    -PHI_WINDOW * 0.98,
    R_MIN + 0.025,
    r"$z_j(\theta_{\rm op})=\lambda_j^{-e^{i\theta_{\rm op}}}$",
    color="white",
    ha="left",
    va="bottom",
    fontsize=10,
    alpha=0.85,
)

# ── Diagnostics ─────────────────────────────────────────────

def local_crowding_score(phi, rad):
    """
    Simple local bunching metric:
    count close pair distances in the displayed window.
    This is not a theorem, just a visual diagnostic.
    """
    visible = (
        (phi >= -PHI_WINDOW) &
        (phi <=  PHI_WINDOW) &
        (rad >= R_MIN) &
        (rad <= R_MAX)
    )

    x = phi[visible]
    y = rad[visible]

    if len(x) < 2:
        return 0, len(x)

    # Normalize axes so distances are comparable.
    xn = x / max(PHI_WINDOW, 1e-12)
    yn = (y - R_MIN) / max(R_MAX - R_MIN, 1e-12)

    pts = np.column_stack([xn, yn])

    close_pairs = 0
    thresh = 0.045

    for a in range(len(pts)):
        d = np.sqrt(np.sum((pts[a+1:] - pts[a])**2, axis=1))
        close_pairs += int(np.sum(d < thresh))

    return close_pairs, len(x)

def lock_error(theta_op):
    """
    Mean distance from unit circle.
    Exact zero at θ=90° and θ=270°.
    """
    phi, rad, _ = zero_positions(theta_op)
    return float(np.mean(np.abs(rad - 1.0)))

# ── Animation update ────────────────────────────────────────

def update(frame_idx):
    theta_op = theta_values[frame_idx]

    phi, rad, _ = zero_positions(theta_op)

    visible = (
        (phi >= -PHI_WINDOW) &
        (phi <=  PHI_WINDOW) &
        (rad >= R_MIN) &
        (rad <= R_MAX)
    )

    if np.any(visible):
        scat.set_offsets(np.column_stack([phi[visible], rad[visible]]))
        scat.set_array(point_colors[visible])
    else:
        scat.set_offsets(np.empty((0, 2)))
        scat.set_array(np.array([]))

    # Trail
    start = max(0, frame_idx - TRAIL_LENGTH)
    trail_phi = all_phi[start:frame_idx+1].reshape(-1)
    trail_r = all_r[start:frame_idx+1].reshape(-1)
    trail_colors = np.tile(point_colors, frame_idx + 1 - start)

    trail_visible = (
        (trail_phi >= -PHI_WINDOW) &
        (trail_phi <=  PHI_WINDOW) &
        (trail_r >= R_MIN) &
        (trail_r <= R_MAX)
    )

    if np.any(trail_visible):
        trail_scat.set_offsets(
            np.column_stack([trail_phi[trail_visible], trail_r[trail_visible]])
        )
        trail_scat.set_array(trail_colors[trail_visible])
    else:
        trail_scat.set_offsets(np.empty((0, 2)))
        trail_scat.set_array(np.array([]))

    deg = np.degrees(theta_op) % 360
    crowd, nvis = local_crowding_score(phi, rad)
    err = lock_error(theta_op)

    title.set_text(
        rf"zero-trajectory resonance plot   |   "
        rf"$N={N_FIXED}$   |   "
        rf"$\theta_{{\rm op}}={deg:.2f}^\circ$   |   "
        rf"visible={nvis}   |   "
        rf"unit-lock error={err:.4f}   |   "
        rf"crowding={crowd}"
    )

    return [scat, trail_scat, title]

ani = animation.FuncAnimation(
    fig,
    update,
    frames=len(theta_values),
    interval=1000 / FPS,
    blit=False,
)

ani.save(
    OUT,
    writer="ffmpeg",
    fps=FPS,
    dpi=DPI,
    bitrate=7000,
)

plt.close(fig)

print(f"Saved -> {OUT}")
HTML(f"""
<video width="900" controls>
  <source src="{OUT}" type="video/mp4">
</video>
""")